# H04 Sell-Price Movement Proxy Features

H01-H03 gave me a useful activation/ranging signal, but not a clean champion. The best activation version improved forecast metrics and overstock, then worsened lost-sales and stockout opportunity cost. That is enough evidence to stop pushing activation mechanics as the main lever and return to the EDA register.

The next hypothesis is sell-price movement. I am being careful with wording here: M5 gives sell prices, but it does not give explicit promotion/ad labels. So this is not a promotion-lift experiment. It is a price-movement proxy experiment.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    for parent in ROOT.parents:
        if (parent / 'backend').exists() and (parent / 'notebooks').exists():
            ROOT = parent
            break
backend_path = ROOT / 'backend'
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))
EDA_JSON = ROOT / 'backend/reports/experiments/manual_ds/00_m5_global_eda/m5_global_eda.json'
PLAN_MD = ROOT / 'backend/reports/experiments/manual_ds/04_price_movement_proxy_h04/price_movement_proxy_h04_plan.md'

with EDA_JSON.open() as fh:
    eda = json.load(fh)

price = eda['price_summary']
display(Markdown(f'Loaded EDA artifact: `{EDA_JSON}`'))

## Data Contract Check

Before this goes into ShelfOps, I want the contract clear. M5 has daily units and sell prices by SKU-store-date. It also has calendar events and SNAP flags. It does not have on-hand inventory, actual shelf availability, supplier lead time, purchase orders, promotion mechanics, ad exposure, display placement, coupon depth, or buyer decisions.

That means I can test whether sell-price movement helps benchmark forecasting and simulated replenishment replay. I cannot claim measured promotion lift or merchant ROI from this dataset.

In [ ]:
price_contract = pd.DataFrame([
    {
        'series_with_price_changes_rate': price['series_with_price_changes_rate'],
        'rows_with_price_change_rate': price['rows_with_price_change_rate'],
        'price_change_rows': price['price_change_rows'],
        'price_drop_rows': price['price_drop_rows'],
        'rows_with_price_drop_rate': price['rows_with_price_drop_rate'],
        'explicit_promo_row_rate': 0.0,
    }
])
display(price_contract)

## What The EDA Suggests

A lot of SKU-store series have at least one sell-price change, but actual price-change rows are sparse. That makes this a good exploratory feature test, not a guaranteed model improvement.

The category reads are mixed too. FOODS does not show the simple price-drop lift pattern in this subset, while HOBBIES and HOUSEHOLD do. That could be a real category behavior difference, or it could be noise from sparse price-change windows. I should not judge this only on global WAPE.

In [ ]:
category_price = pd.DataFrame(price['category_price_effects'])
display(category_price[[
    'category',
    'rows',
    'price_change_row_rate',
    'price_drop_row_rate',
    'price_drop_uplift_vs_no_change',
]])

## H04 Spec Design

The experiment should change one thing: price movement features.

- Keep canonical M5 train/calibration/holdout rows.
- Keep the same core lag, rolling, calendar, product-code, price, and holiday features as the baseline.
- Add `price_change_7` and `price_index_28`.
- Do not use `is_promotional`, because the M5 adapter sets it to zero.
- Do not use `promo_price_interaction`, because without real promo labels that feature is not meaningful.

This is why the ShelfOps spec is `m5_price_movement_proxy_v1`, not the older generic price/promo template.

In [ ]:
h04_spec = {
    'template_id': 'm5_price_movement_proxy_v1',
    'materialized_spec_name': 'm5_price_movement_proxy_h04',
    'experiment_type': 'feature_set',
    'added_features': ['price_change_7', 'price_index_28'],
    'excluded_features': ['is_promotional', 'promo_price_interaction'],
    'claim_boundary': 'Benchmark sell-price movement proxy only. No explicit promo/ad labels or merchant ROI.',
}
display(pd.DataFrame([h04_spec]))

## Success Criteria

I would treat this as useful only if the model improves forecast quality without breaking the replenishment tradeoff.

- WAPE and MASE improve or stay flat globally.
- Bias does not materially worsen.
- Overstock dollars do not increase materially.
- Lost-sales quantity and stockout opportunity cost do not worsen.
- Combined cost proxy does not regress.
- No promotion claim is made unless a future dataset has real promotion exposure and outcome labels.

If this only improves DS metrics while business metrics get worse, I would log it as evidence and move to either segment-specific calibration or intermittent-demand treatment.

## First ShelfOps Run Read

The first H04 run is not a champion, but it is not useless. The price-movement features nudged WAPE/MASE in the right direction and reduced lost-sales pressure, especially on slow/medium segments. The tradeoff is that bias worsened, coverage fell, overstock dollars increased, and the decision replay combined cost proxy got worse.

I also checked whether this holdout actually exercised the hypothesis. It barely did. There were no day-over-day price-change rows in the holdout and only a handful of rows with recent price movement. So I should not call the price family dead from this run; I should call this quick screen mixed and under-exposed.

In [ ]:
REPORT_JSON = ROOT / 'backend/reports/experiments/47050cfd-1ead-4a59-98ef-5d3a6f9c00ba.report.json'
with REPORT_JSON.open() as fh:
    run = json.load(fh)

baseline = run['baseline']['holdout_metrics']
challenger = run['challenger']['holdout_metrics']
metrics = ['wape', 'mase', 'mae', 'bias_pct', 'coverage', 'lost_sales_qty', 'opportunity_cost_stockout', 'overstock_dollars']
rows = []
for metric in metrics:
    b = baseline[metric]
    c = challenger[metric]
    rows.append({
        'metric': metric,
        'champion': b,
        'h04_challenger': c,
        'delta': c - b,
        'pct_delta': ((c - b) / abs(b)) if b else None,
    })
display(pd.DataFrame(rows))

decision = run['decision_replay']['results']
decision_rows = []
for metric in ['service_level', 'stockout_days', 'lost_sales_units', 'overstock_dollars', 'combined_cost_proxy', 'po_count']:
    b = decision['champion'][metric]
    c = decision['challenger'][metric]
    decision_rows.append({'metric': metric, 'champion_policy': b, 'h04_policy': c, 'delta': c - b})
display(pd.DataFrame(decision_rows))

failed_gates = [gate for gate, passed in run['promotion_comparison']['gate_checks'].items() if not passed]
display(pd.DataFrame({'failed_gate': failed_gates}))

In [ ]:
m5_path = ROOT / 'data/benchmarks/m5_walmart/subset_20spc/canonical_transactions.csv'
m5 = pd.read_csv(m5_path, low_memory=False, parse_dates=['date'])
m5 = m5.sort_values(['store_id', 'product_id', 'date']).copy()
series = m5[['store_id', 'product_id']].drop_duplicates().head(60)
m5 = m5.merge(series, on=['store_id', 'product_id'], how='inner')
m5['price'] = pd.to_numeric(m5.get('price', m5.get('sell_price')), errors='coerce')
m5['prior_price_1'] = m5.groupby(['store_id', 'product_id'])['price'].shift(1)
m5['prior_price_7'] = m5.groupby(['store_id', 'product_id'])['price'].shift(7)
m5['price_changed_1'] = (m5['price'] - m5['prior_price_1']).abs().fillna(0) > 1e-9
m5['price_changed_7'] = ((m5['price'] - m5['prior_price_7']) / m5['prior_price_7']).abs().fillna(0) > 1e-9
m5['within_7d_price_change'] = (
    m5.groupby(['store_id', 'product_id'])['price_changed_1']
    .transform(lambda s: s.rolling(8, min_periods=1).max())
    .astype(bool)
)

holdout = m5[(m5['date'] >= pd.Timestamp('2016-03-28')) & (m5['date'] <= pd.Timestamp('2016-04-24'))]
display(pd.DataFrame([{
    'holdout_rows': len(holdout),
    'day_over_day_price_change_rows': int(holdout['price_changed_1'].sum()),
    'price_change_7_rows': int(holdout['price_changed_7'].sum()),
    'within_7d_price_change_rows': int(holdout['within_7d_price_change'].sum()),
    'within_7d_price_change_rate': float(holdout['within_7d_price_change'].mean()),
}]))

## Extended Backtest Read

The follow-up run used the same executable H04 spec, but expanded validation to 180 series and six rolling 28-day windows. This is the cleaner test. It gives the price hypothesis much more exposure, so I can make a firmer call than I could from the quick screen.

The result is still not a champion. The signal is mixed window to window, average WAPE/MASE are essentially flat and slightly worse, coverage weakens, and combined cost is slightly worse. That is enough to stop treating generic price momentum as the next global champion path.

In [ ]:
EXTENDED_JSON = ROOT / 'backend/reports/experiments/514960f9-2496-4177-8aad-fb0418f89dfe.report.json'
with EXTENDED_JSON.open() as fh:
    extended = json.load(fh)

rolling = extended['rolling_validation']
display(pd.DataFrame([{
    'requested_windows': rolling['requested_windows'],
    'completed_windows': rolling['completed_windows'],
    'baseline_avg_wape': rolling['summary_metrics']['baseline_avg_wape'],
    'challenger_avg_wape': rolling['summary_metrics']['challenger_avg_wape'],
    'baseline_avg_mase': rolling['summary_metrics']['baseline_avg_mase'],
    'challenger_avg_mase': rolling['summary_metrics']['challenger_avg_mase'],
    'baseline_avg_combined_cost_proxy': rolling['summary_metrics']['baseline_avg_combined_cost_proxy'],
    'challenger_avg_combined_cost_proxy': rolling['summary_metrics']['challenger_avg_combined_cost_proxy'],
    'temporal_validation_gate': rolling['gate_checks']['temporal_validation_gate'],
}]))

window_rows = []
for window in rolling['windows']:
    window_rows.append({
        'window': window['window'],
        'holdout_start': window['holdout_start'],
        'holdout_end': window['holdout_end'],
        'champion_wape': window['baseline']['wape'],
        'h04_wape': window['challenger']['wape'],
        'wape_delta': window['challenger']['wape'] - window['baseline']['wape'],
        'gates_passed': window['benchmark_gates_passed'],
        'reason': window['reason'],
    })
display(pd.DataFrame(window_rows))

In [ ]:
from ml.decision_experiment import DecisionExperimentConfig, prepare_decision_frame

raw_m5 = pd.read_csv(m5_path, low_memory=False)
prepared = prepare_decision_frame(raw_m5, config=DecisionExperimentConfig(max_rows=250000, max_series=180))
prepared['price'] = pd.to_numeric(prepared['price'], errors='coerce')
prepared['prior_price_1'] = prepared.groupby(['store_id', 'product_id'])['price'].shift(1)
prepared['prior_price_7'] = prepared.groupby(['store_id', 'product_id'])['price'].shift(7)
prepared['price_changed_1'] = (prepared['price'] - prepared['prior_price_1']).abs().fillna(0) > 1e-9
prepared['price_changed_7'] = ((prepared['price'] - prepared['prior_price_7']) / prepared['prior_price_7']).abs().fillna(0) > 1e-9
prepared['within_7d_price_change'] = (
    prepared.groupby(['store_id', 'product_id'])['price_changed_1']
    .transform(lambda s: s.rolling(8, min_periods=1).max())
    .astype(bool)
)

latest = prepared['date'].max()
exposure_rows = []
for idx in range(6):
    end = latest - pd.Timedelta(days=28 * idx)
    start = end - pd.Timedelta(days=27)
    window = prepared[(prepared['date'] >= start) & (prepared['date'] <= end)]
    exposure_rows.append({
        'window': idx + 1,
        'holdout_start': start.date().isoformat(),
        'holdout_end': end.date().isoformat(),
        'rows': len(window),
        'day_over_day_price_change_rows': int(window['price_changed_1'].sum()),
        'price_change_7_rows': int(window['price_changed_7'].sum()),
        'within_7d_price_change_rows': int(window['within_7d_price_change'].sum()),
    })
exposure = pd.DataFrame(exposure_rows)
display(exposure)
display(pd.DataFrame([{
    'total_rows': int(exposure['rows'].sum()),
    'total_day_over_day_price_change_rows': int(exposure['day_over_day_price_change_rows'].sum()),
    'total_price_change_7_rows': int(exposure['price_change_7_rows'].sum()),
    'total_within_7d_price_change_rows': int(exposure['within_7d_price_change_rows'].sum()),
}]))

In [ ]:
display(Markdown(PLAN_MD.read_text()))